## Redrob Hackathon: Reranker and Trap Filter

### What this notebook does
This notebook takes the shortlist from hybrid retrieval and reranks it using deeper role-fit signals, trap checks, and consistency penalties.

### Why this notebook is needed
Retrieval gives us plausible candidates, but not the final best candidates.

We now need to:
- reward strong role fit
- penalize weak or misleading profiles
- detect possible honeypots
- create a more trustworthy ranking before final reasoning and submission

In [12]:
!pip install -q \
pandas \
numpy \
pyarrow \
sentence-transformers \
tqdm

In [13]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import CrossEncoder

In [14]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
PROJECT_DIR = Path("/content/drive/MyDrive/Redrob_Hackathon")
DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"

FEATURES_PATH = ARTIFACTS_DIR / "candidate_features.parquet"
RETRIEVAL_PATH = ARTIFACTS_DIR / "candidate_retrieval_shortlist.parquet"
JD_COMPASS_PATH = ARTIFACTS_DIR / "jd_compass.json"

In [16]:
print("Features exists:", FEATURES_PATH.exists())
print("Retrieval shortlist exists:", RETRIEVAL_PATH.exists())
print("JD compass exists:", JD_COMPASS_PATH.exists())

Features exists: True
Retrieval shortlist exists: True
JD compass exists: True


In [17]:
features_df = pd.read_parquet(FEATURES_PATH)
retrieval_df = pd.read_parquet(RETRIEVAL_PATH)

with open(JD_COMPASS_PATH, "r", encoding="utf-8") as f:
    jd_compass = json.load(f)

print("features_df shape:", features_df.shape)
print("retrieval_df shape:", retrieval_df.shape)

features_df shape: (100000, 53)
retrieval_df shape: (1500, 58)


In [18]:
rerank_df = retrieval_df.merge(
    features_df,
    on="candidate_id",
    how="left",
    suffixes=("_retrieval", "_full")
)

print("rerank_df shape:", rerank_df.shape)
display(rerank_df.head())

rerank_df shape: (1500, 110)


,candidate_id,anonymized_name_retrieval,headline_retrieval,summary_retrieval,location_retrieval,country_retrieval,years_of_experience_retrieval,current_title_retrieval,current_company_retrieval,current_company_size_retrieval,...,availability_strength_full,core_ai_ml_match_count_full,retrieval_ranking_match_count_full,production_engineering_match_count_full,evaluation_quality_match_count_full,seniority_fit_score_full,location_fit_flag_full,flag_possible_framework_only_profile_full,flag_possible_non_ir_ai_profile_full,role_evidence_score_full
0,CAND_0086022,Dhruv Naidu,"Senior Applied Scientist | LLMs, RAG, Vector S...",Senior AI engineer with 5.3 years of hands-on ...,"Kolkata, West Bengal",India,5.3,Senior Applied Scientist,Sarvam AI,51-200,...,86.500000,1,8,1,3,1.0,0,False,False,27.5
1,CAND_0055905,Anika Rao,"Senior Machine Learning Engineer | LLMs, RAG, ...",Senior AI engineer with 8.1 years of hands-on ...,London,UK,8.1,Senior Machine Learning Engineer,Flipkart,10001+,...,47.712903,3,9,1,3,1.0,0,False,False,31.5
2,CAND_0046525,Tanvi Mukherjee,Senior Machine Learning Engineer | Building AI...,Senior AI engineer with 6.1 years of hands-on ...,"Pune, Maharashtra",India,6.1,Senior Machine Learning Engineer,Genpact AI,10001+,...,47.219672,2,7,1,3,1.0,1,False,False,27.0
3,CAND_0005649,Riya Kapoor,"Senior Data Scientist | Search, Ranking & Retr...",Machine learning engineer with 7.4 years of ex...,"Delhi, Delhi",India,7.4,Senior Data Scientist,Sarvam AI,51-200,...,37.649451,2,8,2,4,1.0,1,False,False,32.5
4,CAND_0069905,Nisha Bansal,"Applied ML Engineer | Search, Ranking & Retrieval",Machine learning engineer with 6.6 years of ex...,"Bhubaneswar, Odisha",India,6.6,Applied ML Engineer,Sarvam AI,51-200,...,43.949451,2,8,2,4,1.0,0,False,False,32.0


In [19]:
print(retrieval_df.shape)
print(rerank_df.shape)

(1500, 58)
(1500, 110)


In [21]:
print([c for c in rerank_df.columns if "profile_text" in c or "skill_text" in c])

['skill_text_retrieval', 'profile_text_retrieval', 'skill_text_full', 'profile_text_full']


In [22]:
# Cross encoder reranking setup
jd_text = (
    jd_compass["jd_summary"]["role_title"] + ". " +
    jd_compass["jd_summary"]["role_goal"] + ". " +
    "Must have: " + ", ".join(jd_compass["jd_summary"]["must_have_skills"]) + ". " +
    "Preferred: " + ", ".join(jd_compass["jd_summary"]["strong_preferred_skills"])
)
# This builds one long job-description string having role title, role goal, must have skills and prefered skills


rerank_df["rerank_candidate_text"] = (
    rerank_df["profile_text_full"].fillna("") + " " +
    rerank_df["skill_text_full"].fillna("")
)
# This creates one combined text per candidate from full profile text and full skill text
# while fillna("") prevents NaN from breaking string concatenation

print(jd_text[:1000])
print("\nCandidate text example:\n")
print(rerank_df["rerank_candidate_text"].iloc[0][:1000])

Senior AI Engineer. Build and ship practical AI systems involving retrieval, ranking, LLMs, evaluation, and production engineering.. Must have: python, machine learning, llm, embeddings, retrieval, vector search, ranking, evaluation. Preferred: rag, fine tuning, transformers, pytorch, faiss, elasticsearch, docker, aws, gcp, langchain, langgraph

Candidate text example:

Senior Applied Scientist | LLMs, RAG, Vector Search | ex-Top Tech Senior AI engineer with 5.3 years of hands-on experience building production ML systems, with a focus on search, retrieval, and ranking. Most recently, I led the migration from keyword-based ranking to a learning-to-rank model with embedded behavioral signals, handling peak QPS of 8K with sub-200ms p95. My day-to-day work spans embedding model selection and fine-tuning, hybrid retrieval architecture, learning-to-rank, behavioral-signal integration, and the offline/online evaluation that ties it all together. I've shipped systems in both early-stage produc

In [23]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder loaded.


In [24]:
pairs = [
    (jd_text, text)
    for text in rerank_df["rerank_candidate_text"].tolist()
]
# This creates a list of tuples and each tuple is (job_description_text, candidate_text)
# This prepares the input for the crossencoder the JD never changes only the candidate profile changes
print("Total pairs:", len(pairs))

Total pairs: 1500


In [38]:
# Because a Cross Encoder always compares two texts together the input format is:
# (Text A, Text B) or (Sentence A, Sentence B)
rerank_scores = cross_encoder.predict(
    pairs,
    batch_size=32,
    show_progress_bar=True
)

rerank_df["cross_encoder_score"] = rerank_scores

display(
    rerank_df[
        [
            "candidate_id",
            "current_title_retrieval",
            "years_of_experience_full",
            "cross_encoder_score"
        ]
    ].head(20)
)

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

,candidate_id,current_title_retrieval,years_of_experience_full,cross_encoder_score
0,CAND_0055905,Senior Machine Learning Engineer,8.1,4.306592
1,CAND_0005649,Senior Data Scientist,7.4,4.944154
2,CAND_0081846,Lead AI Engineer,6.7,5.027650
3,CAND_0086022,Senior Applied Scientist,5.3,3.959594
4,CAND_0069905,Applied ML Engineer,6.6,2.990727
5,CAND_0046525,Senior Machine Learning Engineer,6.1,1.526658
6,CAND_0077337,Staff Machine Learning Engineer,7.0,4.429589
7,CAND_0044883,AI Engineer,6.3,4.432701
8,CAND_0083307,Search Engineer,7.8,4.491690
9,CAND_0079387,AI Engineer,6.9,2.524641


In [39]:
rerank_df["cross_encoder_score"].describe()

,cross_encoder_score
count,1500.000000
mean,-1.009054
std,2.676007
min,-7.476154
25%,-2.968798
50%,-0.744180
75%,1.071425
max,5.062526


Trap filter

In [27]:
rerank_df["flag_extreme_experience_mismatch"] = (
    (rerank_df["years_of_experience_full"] > 12) &
    (rerank_df["seniority_fit_score_full"] == 0)
)

rerank_df["flag_weak_role_evidence"] = (
    rerank_df["role_evidence_score_full"] < 2.5
)

rerank_df["flag_high_risk_profile"] = (
    rerank_df["total_risk_score_full"] >= 4
)

rerank_df["flag_inactive_candidate"] = (
    (rerank_df["open_to_work_flag_full"] == 0) &
    (rerank_df["recruiter_response_rate_full"] < 0.25) &
    (rerank_df["avg_response_time_hours_full"] > 150)
)

In [28]:
rerank_df["flag_possible_framework_only_profile"] = (
    rerank_df["flag_possible_framework_only_profile_full"].fillna(False)
)

rerank_df["flag_possible_non_ir_ai_profile"] = (
    rerank_df["flag_possible_non_ir_ai_profile_full"].fillna(False)
)

In [29]:
rerank_df["trap_score"] = (
    rerank_df["flag_extreme_experience_mismatch"].astype(int) * 2.0 +
    rerank_df["flag_weak_role_evidence"].astype(int) * 1.5 +
    rerank_df["flag_high_risk_profile"].astype(int) * 2.0 +
    rerank_df["flag_inactive_candidate"].astype(int) * 1.5 +
    rerank_df["flag_possible_framework_only_profile"].astype(int) * 1.5 +
    rerank_df["flag_possible_non_ir_ai_profile"].astype(int) * 1.5
)

display(
    rerank_df[
        [
            "candidate_id",
            "trap_score",
            "flag_extreme_experience_mismatch",
            "flag_weak_role_evidence",
            "flag_high_risk_profile",
            "flag_inactive_candidate"
        ]
    ].head(20)
)

,candidate_id,trap_score,flag_extreme_experience_mismatch,flag_weak_role_evidence,flag_high_risk_profile,flag_inactive_candidate
0,CAND_0086022,0.0,False,False,False,False
1,CAND_0055905,0.0,False,False,False,False
2,CAND_0046525,0.0,False,False,False,False
3,CAND_0005649,0.0,False,False,False,False
4,CAND_0069905,0.0,False,False,False,False
5,CAND_0079387,0.0,False,False,False,False
6,CAND_0081846,0.0,False,False,False,False
7,CAND_0055992,2.0,True,False,False,False
8,CAND_0046064,0.0,False,False,False,False
9,CAND_0040887,0.0,False,False,False,False


In [30]:
rerank_df["final_rerank_score"] = (
    rerank_df["cross_encoder_score"] * 3.0 +
    rerank_df["role_evidence_score_full"] * 1.5 +
    rerank_df["retrieval_stage_score"] * 1.0 +
    rerank_df["availability_strength_full"] * 0.02 +
    rerank_df["seniority_fit_score_full"] * 0.75 -
    rerank_df["trap_score"] * 2.5 -
    rerank_df["total_risk_score_full"] * 0.5
)

display(
    rerank_df[
        [
            "candidate_id",
            "current_title_retrieval",
            "cross_encoder_score",
            "role_evidence_score_full",
            "trap_score",
            "total_risk_score_full",
            "final_rerank_score"
        ]
    ].head(20)
)

,candidate_id,current_title_retrieval,cross_encoder_score,role_evidence_score_full,trap_score,total_risk_score_full,final_rerank_score
0,CAND_0086022,Senior Applied Scientist,4.306592,27.5,0.0,2,72.848459
1,CAND_0055905,Senior Machine Learning Engineer,4.944154,31.5,0.0,1,78.016651
2,CAND_0046525,Senior Machine Learning Engineer,5.027650,27.0,0.0,0,71.782399
3,CAND_0005649,Senior Data Scientist,3.959594,32.5,0.0,0,76.579591
4,CAND_0069905,Applied ML Engineer,2.990727,32.0,0.0,0,72.572914
5,CAND_0079387,AI Engineer,1.526658,32.0,0.0,2,67.169423
6,CAND_0081846,Lead AI Engineer,4.429589,31.5,0.0,2,74.986441
7,CAND_0055992,AI Engineer,4.432701,27.0,2.0,0,63.126666
8,CAND_0046064,Senior NLP Engineer,4.491690,25.5,0.0,0,66.829018
9,CAND_0040887,Machine Learning Engineer,2.524641,26.8,0.0,0,62.642114


In [31]:
rerank_df = rerank_df.sort_values(
    ["final_rerank_score", "cross_encoder_score", "candidate_id"],
    ascending=[False, False, True]
).reset_index(drop=True)

display(
    rerank_df[
        [
            "candidate_id",
            "current_title_retrieval",
            "years_of_experience_full",
            "final_rerank_score",
            "cross_encoder_score",
            "role_evidence_score_full",
            "trap_score",
            "total_risk_score_full"
        ]
    ].head(30)
)

,candidate_id,current_title_retrieval,years_of_experience_full,final_rerank_score,cross_encoder_score,role_evidence_score_full,trap_score,total_risk_score_full
0,CAND_0055905,Senior Machine Learning Engineer,8.1,78.016651,4.944154,31.5,0.0,1
1,CAND_0005649,Senior Data Scientist,7.4,76.579591,3.959594,32.5,0.0,0
2,CAND_0081846,Lead AI Engineer,6.7,74.986441,4.429589,31.5,0.0,2
3,CAND_0086022,Senior Applied Scientist,5.3,72.848459,4.306592,27.5,0.0,2
4,CAND_0069905,Applied ML Engineer,6.6,72.572914,2.990727,32.0,0.0,0
5,CAND_0046525,Senior Machine Learning Engineer,6.1,71.782399,5.027650,27.0,0.0,0
6,CAND_0077337,Staff Machine Learning Engineer,7.0,70.739877,4.755748,28.5,0.0,2
7,CAND_0044883,AI Engineer,6.3,69.618009,3.680589,30.0,0.0,0
8,CAND_0083307,Search Engineer,7.8,67.680995,2.118656,32.0,0.0,3
9,CAND_0079387,AI Engineer,6.9,67.169423,1.526658,32.0,0.0,2


In [32]:
top_candidates = rerank_df.head(200).copy()

display(
    top_candidates[
        [
            "candidate_id",
            "current_title_retrieval",
            "current_company_retrieval",
            "location_full",
            "years_of_experience_full",
            "final_rerank_score",
            "cross_encoder_score",
            "role_evidence_score_full",
            "trap_score",
            "total_risk_score_full"
        ]
    ].head(30)
)

,candidate_id,current_title_retrieval,current_company_retrieval,location_full,years_of_experience_full,final_rerank_score,cross_encoder_score,role_evidence_score_full,trap_score,total_risk_score_full
0,CAND_0055905,Senior Machine Learning Engineer,Flipkart,London,8.1,78.016651,4.944154,31.5,0.0,1
1,CAND_0005649,Senior Data Scientist,Sarvam AI,"Delhi, Delhi",7.4,76.579591,3.959594,32.5,0.0,0
2,CAND_0081846,Lead AI Engineer,Razorpay,"Jaipur, Rajasthan",6.7,74.986441,4.429589,31.5,0.0,2
3,CAND_0086022,Senior Applied Scientist,Sarvam AI,"Kolkata, West Bengal",5.3,72.848459,4.306592,27.5,0.0,2
4,CAND_0069905,Applied ML Engineer,Sarvam AI,"Bhubaneswar, Odisha",6.6,72.572914,2.990727,32.0,0.0,0
5,CAND_0046525,Senior Machine Learning Engineer,Genpact AI,"Pune, Maharashtra",6.1,71.782399,5.027650,27.0,0.0,0
6,CAND_0077337,Staff Machine Learning Engineer,Paytm,"Kochi, Kerala",7.0,70.739877,4.755748,28.5,0.0,2
7,CAND_0044883,AI Engineer,Mad Street Den,Berlin,6.3,69.618009,3.680589,30.0,0.0,0
8,CAND_0083307,Search Engineer,CRED,"Vizag, Andhra Pradesh",7.8,67.680995,2.118656,32.0,0.0,3
9,CAND_0079387,AI Engineer,Microsoft,"Trivandrum, Kerala",6.9,67.169423,1.526658,32.0,0.0,2


In [33]:
FINAL_SHORTLIST_SIZE = 100

final_shortlist_df = rerank_df.head(FINAL_SHORTLIST_SIZE).copy()

print("Final shortlist size:", final_shortlist_df.shape[0])

display(
    final_shortlist_df[
        [
            "candidate_id",
            "current_title_retrieval",
            "current_company_retrieval",
            "years_of_experience_full",
            "final_rerank_score",
            "cross_encoder_score",
            "trap_score"
        ]
    ]
)

Final shortlist size: 100


,candidate_id,current_title_retrieval,current_company_retrieval,years_of_experience_full,final_rerank_score,cross_encoder_score,trap_score
0,CAND_0055905,Senior Machine Learning Engineer,Flipkart,8.1,78.016651,4.944154,0.0
1,CAND_0005649,Senior Data Scientist,Sarvam AI,7.4,76.579591,3.959594,0.0
2,CAND_0081846,Lead AI Engineer,Razorpay,6.7,74.986441,4.429589,0.0
3,CAND_0086022,Senior Applied Scientist,Sarvam AI,5.3,72.848459,4.306592,0.0
4,CAND_0069905,Applied ML Engineer,Sarvam AI,6.6,72.572914,2.990727,0.0
...,...,...,...,...,...,...,...
95,CAND_0060315,Senior Software Engineer (ML),Glance,5.4,48.258766,3.152005,0.0
96,CAND_0075439,Machine Learning Engineer,Flipkart,4.3,48.211474,3.176886,0.0
97,CAND_0016163,Applied ML Engineer,Dream11,6.7,47.781790,0.908126,0.0
98,CAND_0011432,Senior Data Scientist,Amazon,7.6,47.755714,2.590059,0.0


In [34]:
RERANK_PATH = ARTIFACTS_DIR / "reranked_candidates.parquet"
RERANK_CSV_PATH = ARTIFACTS_DIR / "reranked_candidates.csv"
SHORTLIST_PATH = ARTIFACTS_DIR / "final_shortlist_for_reasoning.parquet"

rerank_df.to_parquet(RERANK_PATH, index=False)
rerank_df.to_csv(RERANK_CSV_PATH, index=False)
final_shortlist_df.to_parquet(SHORTLIST_PATH, index=False)

print("Saved:", RERANK_PATH)
print("Saved:", RERANK_CSV_PATH)
print("Saved:", SHORTLIST_PATH)

Saved: /content/drive/MyDrive/Redrob_Hackathon/artifacts/reranked_candidates.parquet
Saved: /content/drive/MyDrive/Redrob_Hackathon/artifacts/reranked_candidates.csv
Saved: /content/drive/MyDrive/Redrob_Hackathon/artifacts/final_shortlist_for_reasoning.parquet


In [35]:
rerank_summary = {
    "retrieval_pool_size": int(rerank_df.shape[0]),
    "final_shortlist_size": int(final_shortlist_df.shape[0]),
    "top_final_score": float(rerank_df["final_rerank_score"].max()),
    "embedding_reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2"
}

with open(ARTIFACTS_DIR / "rerank_summary.json", "w", encoding="utf-8") as f:
    json.dump(rerank_summary, f, indent=4)

print(rerank_summary)

{'retrieval_pool_size': 1500, 'final_shortlist_size': 100, 'top_final_score': 78.01665068381078, 'embedding_reranker_model': 'cross-encoder/ms-marco-MiniLM-L-6-v2'}
